# Notebook 9 — Regression Visualisations

**Pipeline position:** runs after `6_Regressions_Unified.ipynb`  
**Inputs:** `intermediary/Master.csv`, `intermediary/clustersagg.csv`  
**Outputs:** `Visualisation/charts/regression/`  

Charts produced:
- Chart 16: Coefficient forest plot (Models 3a–3e)
- Chart 17: R² comparison across models
- Chart 18: HCI × NR Production interaction scatter
- Chart 19: NR rents vs ECI by cluster
- Chart 20: Predicted vs Actual ECI (Model 3b)

# Regression Visualisations

**Capstone — Moody's Ratings**  
*Industrial Upgrading in Emerging Markets with Rich Natural Resources*

Charts for Section 4 (Regression Analysis). Runs Models 3a–3e with country-clustered SE.  
Reads `intermediary/Master.csv`. Saves to `Visualisation/charts/regression/`.

Charts produced:
- **16** — Coefficient forest plot (Models 3a–3e)
- **17** — R² comparison across models
- **18** — HCI × NR Production interaction scatter
- **19** — NR rents vs ECI by cluster (scatter)
- **20** — Predicted vs Actual ECI (Model 3b)

## 0. Setup

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Project root ──────────────────────────────────────────────────────────────
def _find_root(marker='intermediary'):
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, marker)):
            return d
        d = os.path.dirname(d)
    fallback = '/Users/leoss/Desktop/GitHub/Capstone/CLEAN'
    if os.path.isdir(os.path.join(fallback, marker)):
        return fallback
    raise RuntimeError("Could not find project root")

ROOT = _find_root()
os.chdir(ROOT)

OUT = os.path.join(ROOT, 'Visualisation', 'charts', 'regression')
os.makedirs(OUT, exist_ok=True)

# ── Style ─────────────────────────────────────────────────────────────────────
FONT = 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
CFG  = {'displayModeBar': False, 'responsive': True}

SPEC_COLORS = {
    '3a': '#4a6fa5', '3b': '#c23a3a', '3c': '#2e7d4a',
    '3d': '#d4853b', '3e': '#7b6fa5',
}
CLUSTER_COLORS = {
    'Petrostates':           '#E63946',
    'Oil Exporters':         '#457B9D',
    'Diversified Exporters': '#2A9D8F',
    'Gold & Coal':           '#E9C46A',
}

def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=60, r=40, t=50, b=60))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    fig.write_html(f"{path}.html", config=CFG)
    print(f"  Saved: {path}.html")
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=3)
        print(f"  Saved: {path}.png")
    except Exception:
        print(f"  (PNG skipped — pip install kaleido)")

print(f'Root: {ROOT}')
print(f'Output: {OUT}')


## 1. Load Data & Feature Engineering

In [ ]:
ECI_COL = 'Economic Complexity Index'

master   = pd.read_csv('intermediary/Master.csv', dtype={'Country Code': str})
clusters = pd.read_csv('intermediary/clustersagg.csv', dtype={'Country Code': str})

cl_map = clusters[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')
df = master.merge(cl_map, on='Country Code', how='inner')
df['Year'] = df['Year'].astype(int)
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

# ── Feature engineering ───────────────────────────────────────────────────────
df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population'].replace(0, np.nan))

df['log_HCI']              = np.log1p(df['Human capital index'].clip(lower=0))
df['log_GFCF']             = np.log1p(
    df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].clip(lower=0))
df['log_Production_Value'] = np.log1p(df['Total_Production_Value_Per_Capita'].clip(lower=0))

df['ECI_lag1']  = df.groupby('Country Code')[ECI_COL].shift(1)
df['delta_ECI'] = df[ECI_COL] - df['ECI_lag1']

BASE_INDEP = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate',
    'Rule of law index',
    'log_Production_Value',
    'Trade (% of GDP)',
]
EXTRA_CONTROLS = [
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]

for var in BASE_INDEP:
    df[f'{var}_lag1'] = df.groupby('Country Code')[var].shift(1)

hci_c  = df['log_HCI']             - df['log_HCI'].mean()
gfcf_c = df['log_GFCF']            - df['log_GFCF'].mean()
prod_c = df['log_Production_Value'] - df['log_Production_Value'].mean()
df['log_HCI_x_log_Production']  = hci_c  * prod_c
df['log_GFCF_x_log_Production'] = gfcf_c * prod_c

hci_l_c  = df['log_HCI_lag1']             - df['log_HCI_lag1'].mean()
gfcf_l_c = df['log_GFCF_lag1']            - df['log_GFCF_lag1'].mean()
prod_l_c = df['log_Production_Value_lag1'] - df['log_Production_Value_lag1'].mean()
df['log_HCI_x_log_Production_lag1']  = hci_l_c  * prod_l_c
df['log_GFCF_x_log_Production_lag1'] = gfcf_l_c * prod_l_c

DISPLAY_LABELS = {
    'const':                                           'Constant',
    'log_HCI':                                         'Human Capital (log)',
    'log_GFCF':                                        'GFCF (log)',
    'Political stability \u2014 estimate':            'Political Stability',
    'Rule of law index':                               'Rule of Law',
    'log_Production_Value':                            'NR Production (log, pc)',
    'Trade (% of GDP)':                                'Trade (% GDP)',
    'log_HCI_x_log_Production':                        'HCI \u00d7 Production',
    'log_GFCF_x_log_Production':                       'GFCF \u00d7 Production',
    'ECI_lag1':                                        'ECI (t-1)',
    'log_HCI_lag1':                                    'Human Capital (t-1)',
    'log_GFCF_lag1':                                   'GFCF (t-1)',
    'Political stability \u2014 estimate_lag1':       'Political Stability (t-1)',
    'Rule of law index_lag1':                          'Rule of Law (t-1)',
    'log_Production_Value_lag1':                       'NR Production (t-1)',
    'Trade (% of GDP)_lag1':                           'Trade (t-1)',
    'log_HCI_x_log_Production_lag1':                   'HCI \u00d7 Production (t-1)',
    'log_GFCF_x_log_Production_lag1':                  'GFCF \u00d7 Production (t-1)',
    'Hydrocarbons_Dominant':                           'Hydrocarbons dominant',
    'Subsoil_Metals_Dominant':                         'Subsoil metals dominant',
    'Precious_Metals_Dominant':                        'Precious metals dominant',
    'Access to electricity (% of population)':        'Electricity access',
}

print(f'Sample: {df["Country Code"].nunique()} countries, {len(df):,} obs')
print(f'Years: {df["Year"].min()}\u2013{df["Year"].max()}')
print(f'Cluster distribution:')
print(df.drop_duplicates("Country Code")["ClusterLabels"].value_counts().to_string())


## 2. Run Regression Models (3a–3e)

In [ ]:
def run_ols(df_in, dv, regressors, cluster_by='Country Code'):
    req = [dv] + regressors + [cluster_by]
    sub = df_in.dropna(subset=req).copy()
    X   = sm.add_constant(sub[regressors])
    y   = sub[dv]
    res = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': sub[cluster_by]})
    return res, sub

def coef_table(res):
    tbl = pd.DataFrame({
        'Variable': res.params.index,
        'Coef':     res.params.values,
        'SE':       res.bse.values,
        'p':        res.pvalues.values,
        'CI_lo':    res.conf_int().iloc[:, 0].values,
        'CI_hi':    res.conf_int().iloc[:, 1].values,
    })
    tbl['Stars'] = tbl['p'].apply(
        lambda p: '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else '')
    return tbl

ALL_RESULTS = {}

# 3a: Base (no lag)
VARS_3A = BASE_INDEP + ['log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
res_3a, sub_3a = run_ols(df, ECI_COL, VARS_3A)
ALL_RESULTS['3a'] = (res_3a, coef_table(res_3a), sub_3a)
print(f'3a  N={int(res_3a.nobs):,}  R\u00b2={res_3a.rsquared:.4f}')

# 3b: Base + ECI lag
VARS_3B = BASE_INDEP + ['ECI_lag1', 'log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
res_3b, sub_3b = run_ols(df, ECI_COL, VARS_3B)
ALL_RESULTS['3b'] = (res_3b, coef_table(res_3b), sub_3b)
print(f'3b  N={int(res_3b.nobs):,}  R\u00b2={res_3b.rsquared:.4f}')

# 3c: All regressors lagged
LAGGED_BASE = [f'{v}_lag1' for v in BASE_INDEP]
VARS_3C = LAGGED_BASE + ['ECI_lag1', 'log_HCI_x_log_Production_lag1', 'log_GFCF_x_log_Production_lag1']
res_3c, sub_3c = run_ols(df, ECI_COL, VARS_3C)
ALL_RESULTS['3c'] = (res_3c, coef_table(res_3c), sub_3c)
print(f'3c  N={int(res_3c.nobs):,}  R\u00b2={res_3c.rsquared:.4f}')

# 3d: Extended controls + lag
VARS_3D = BASE_INDEP + EXTRA_CONTROLS + ['ECI_lag1', 'log_HCI_x_log_Production', 'log_GFCF_x_log_Production']
VARS_3D = [v for v in VARS_3D if v in df.columns]
res_3d, sub_3d = run_ols(df, ECI_COL, VARS_3D)
ALL_RESULTS['3d'] = (res_3d, coef_table(res_3d), sub_3d)
print(f'3d  N={int(res_3d.nobs):,}  R\u00b2={res_3d.rsquared:.4f}')

# 3e: First differences
res_3e, sub_3e = run_ols(df, 'delta_ECI', VARS_3D)
ALL_RESULTS['3e'] = (res_3e, coef_table(res_3e), sub_3e)
print(f'3e  N={int(res_3e.nobs):,}  R\u00b2={res_3e.rsquared:.4f}')


## Chart 16 — Coefficient Forest Plot (Models 3a–3e)

In [ ]:
FOREST_VARS = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate', 'Rule of law index',
    'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    'ECI_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]

MODEL_ORDER  = ['3a', '3b', '3c', '3d', '3e']
MODEL_LABELS = {
    '3a': '3a Base', '3b': '3b + Lag',
    '3c': '3c All Lagged', '3d': '3d Extended', '3e': '3e \u0394ECI',
}

y_labels = [DISPLAY_LABELS.get(v, v) for v in FOREST_VARS]
y_pos    = {lbl: i for i, lbl in enumerate(y_labels)}
n_specs  = len(MODEL_ORDER)
offsets  = np.linspace(-0.22, 0.22, n_specs)

fig16 = go.Figure()
fig16.add_vline(x=0, line=dict(color='#aab0ba', width=1.5, dash='dash'))

# Horizontal reference bands
for i in range(len(y_labels)):
    if i % 2 == 0:
        fig16.add_hrect(y0=i - 0.48, y1=i + 0.48,
                        fillcolor='rgba(230,235,242,0.35)', line=dict(width=0), layer='below')

for i, m in enumerate(MODEL_ORDER):
    if m not in ALL_RESULTS: continue
    _, tbl, _ = ALL_RESULTS[m]
    col = SPEC_COLORS[m]
    lbl = MODEL_LABELS[m]

    matched = tbl[tbl['Variable'].isin(FOREST_VARS)].copy()
    matched['Label'] = matched['Variable'].map(lambda v: DISPLAY_LABELS.get(v, v))
    matched['y_pos'] = matched['Label'].map(y_pos).fillna(-1) + offsets[i]
    matched = matched[matched['y_pos'] >= 0]

    fig16.add_trace(go.Scatter(
        x=matched['Coef'], y=matched['y_pos'],
        mode='markers',
        marker=dict(size=9, color=col, line=dict(color='white', width=1.5)),
        error_x=dict(
            type='data', symmetric=False,
            array=(matched['CI_hi'] - matched['Coef']).values,
            arrayminus=(matched['Coef'] - matched['CI_lo']).values,
            color=col, thickness=2.0, width=5,
        ),
        name=lbl,
        text=matched['Label'],
        hovertemplate='<b>%{text}</b><br>\u03b2 = %{x:.4f}<extra>' + lbl + '</extra>',
    ))

fig16.update_layout(**base_layout(
    height=max(560, len(y_labels) * 50 + 150),
    margin=dict(l=220, r=60, t=55, b=60),
    xaxis=dict(title='Coefficient (95% CI, SE clustered by country)',
               gridcolor=GRID, gridwidth=0.5, zeroline=False),
    yaxis=dict(
        tickvals=list(y_pos.values()),
        ticktext=list(y_pos.keys()),
        tickfont=dict(size=11), showgrid=True, gridcolor=GRID, gridwidth=0.5,
        range=[-0.55, len(y_labels) - 0.45],
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig16, '16_reg_coefficient_forest', w=1200, h=max(560, len(y_labels) * 50 + 150))
fig16.show(config=CFG)


## Chart 17 — R\u00b2 Comparison Across Models

In [ ]:
perf_rows = []
for m in MODEL_ORDER:
    if m not in ALL_RESULTS: continue
    res, _, sub = ALL_RESULTS[m]
    dv = 'delta_ECI' if m == '3e' else ECI_COL
    y_hat = res.predict(sm.add_constant(sub[res.params.index.drop('const')]))
    perf_rows.append({
        'Model': MODEL_LABELS[m],
        'R\u00b2': round(res.rsquared, 4),
        'R\u00b2 adj': round(res.rsquared_adj, 4),
        'N': int(res.nobs),
        'color': SPEC_COLORS[m],
    })
perf_df = pd.DataFrame(perf_rows)

fig17 = go.Figure()
fig17.add_trace(go.Bar(
    x=perf_df['Model'], y=perf_df['R\u00b2'],
    marker=dict(color=[SPEC_COLORS[m] for m in MODEL_ORDER], opacity=0.88,
                line=dict(color='white', width=1.5)),
    text=[f'{v:.3f}' for v in perf_df['R\u00b2']], textposition='outside',
    textfont=dict(size=11),
    hovertemplate='<b>%{x}</b><br>R\u00b2 = %{y:.4f}<extra></extra>',
    name='R\u00b2',
))
fig17.add_trace(go.Bar(
    x=perf_df['Model'], y=perf_df['R\u00b2 adj'],
    marker=dict(color=[SPEC_COLORS[m] for m in MODEL_ORDER], opacity=0.4,
                pattern=dict(shape='/'), line=dict(color='white', width=1.5)),
    text=[f'{v:.3f}' for v in perf_df['R\u00b2 adj']], textposition='outside',
    textfont=dict(size=9),
    hovertemplate='<b>%{x}</b><br>R\u00b2 adj = %{y:.4f}<extra></extra>',
    name='R\u00b2 adj',
))

# Add N annotations below bars
for _, row in perf_df.iterrows():
    fig17.add_annotation(
        x=row['Model'], y=-0.06, text=f"N={row['N']:,}", showarrow=False,
        font=dict(size=9, color='#666'), yref='y',
    )

fig17.update_layout(**base_layout(
    height=480, barmode='group',
    margin=dict(l=60, r=40, t=55, b=80),
    xaxis=dict(tickfont=dict(size=12)),
    yaxis=dict(title='R\u00b2', range=[0, 1.12], gridcolor=GRID, gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig17, '17_reg_r2_comparison', w=900, h=480)
fig17.show(config=CFG)


## Chart 18 — HCI \u00d7 NR Production Interaction

In [ ]:
plot_df = df[['Country Code', 'Country Name', 'Year', ECI_COL,
              'log_HCI', 'log_Production_Value', 'ClusterLabels']].dropna().copy()

# Quartile of NR production value
plot_df['prod_q'] = pd.qcut(plot_df['log_Production_Value'], 4,
                            labels=['Q1 (low NR)', 'Q2', 'Q3', 'Q4 (high NR)'])
PROD_COLORS = {'Q1 (low NR)': '#a8c5da', 'Q2': '#4a6fa5',
               'Q3': '#d4853b', 'Q4 (high NR)': '#c23a3a'}

fig18 = go.Figure()
for q, col in PROD_COLORS.items():
    sub = plot_df[plot_df['prod_q'] == q]
    fig18.add_trace(go.Scatter(
        x=sub['log_HCI'], y=sub[ECI_COL], mode='markers',
        marker=dict(size=5, color=col, opacity=0.65, line=dict(color='white', width=0.5)),
        name=q,
        customdata=np.stack([sub['Country Code'], sub['Country Name'],
                             sub['Year'], sub['log_Production_Value']], axis=1),
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]}, %{customdata[2]})<br>'
                      'log HCI: %{x:.3f}<br>ECI: %{y:.3f}<br>'
                      'log Prod: %{customdata[3]:.3f}<extra>' + q + '</extra>',
    ))

# OLS trendline per quartile
for q, col in PROD_COLORS.items():
    sub = plot_df[plot_df['prod_q'] == q].copy()
    if len(sub) < 10: continue
    xv = sub['log_HCI'].values
    yv = sub[ECI_COL].values
    c  = np.polyfit(xv, yv, 1)
    xl = np.linspace(xv.min(), xv.max(), 80)
    fig18.add_trace(go.Scatter(
        x=xl, y=np.polyval(c, xl), mode='lines',
        line=dict(color=col, width=2, dash='dot'),
        showlegend=False, hoverinfo='skip',
    ))

fig18.update_layout(**base_layout(
    height=560,
    margin=dict(l=70, r=40, t=55, b=60),
    xaxis=dict(title='Human Capital Index (log)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(title='NR Production quartile', font=dict(size=11),
                bgcolor='rgba(255,255,255,0.9)', bordercolor=GRID, borderwidth=1),
))
save(fig18, '18_reg_hci_production_interaction', w=1100, h=560)
fig18.show(config=CFG)


## Chart 19 — NR Rents vs ECI by Cluster

In [ ]:
scat_df = df[['Country Code', 'Country Name', 'Year', ECI_COL,
              'Total natural resources rents (% of GDP)', 'ClusterLabels']].dropna().copy()
scat_df.rename(columns={'Total natural resources rents (% of GDP)': 'NR_Rents'}, inplace=True)

years_available = sorted(scat_df['Year'].unique())
year_marks = [1995, 2000, 2005, 2010, 2015, 2019]

fig19 = go.Figure()
for lbl in sorted(CLUSTER_COLORS.keys()):
    sub = scat_df[scat_df['ClusterLabels'] == lbl]
    if len(sub) == 0: continue
    color = CLUSTER_COLORS[lbl]
    fig19.add_trace(go.Scatter(
        x=sub['NR_Rents'], y=sub[ECI_COL], mode='markers',
        marker=dict(size=5, color=color, opacity=0.6, line=dict(color='white', width=0.4)),
        name=lbl,
        customdata=np.stack([sub['Country Code'], sub['Country Name'], sub['Year']], axis=1),
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]}, %{customdata[2]})<br>'
                      'NR Rents: %{x:.1f}%<br>ECI: %{y:.3f}<extra>' + lbl + '</extra>',
    ))

# Overall OLS trendline
xv = scat_df['NR_Rents'].values
yv = scat_df[ECI_COL].values
c  = np.polyfit(xv, yv, 1)
xl = np.linspace(xv.min(), xv.max(), 100)
fig19.add_trace(go.Scatter(
    x=xl, y=np.polyval(c, xl), mode='lines',
    line=dict(color=NAVY, width=2, dash='dash'),
    name=f'OLS trend (\u03b2={c[0]:.4f})',
    hoverinfo='skip',
))

fig19.update_layout(**base_layout(
    height=560,
    margin=dict(l=70, r=40, t=55, b=60),
    xaxis=dict(title='Total NR Rents (% of GDP)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
))
save(fig19, '19_reg_nr_rents_vs_eci', w=1100, h=560)
fig19.show(config=CFG)


## Chart 20 — Predicted vs Actual ECI (Model 3b)

In [ ]:
res3b, tbl3b, sub3b = ALL_RESULTS['3b']
y_actual = sub3b[ECI_COL].values
y_pred   = res3b.fittedvalues.values
resid    = y_actual - y_pred

lims = [min(y_actual.min(), y_pred.min()) - 0.15,
        max(y_actual.max(), y_pred.max()) + 0.15]

fig20 = go.Figure()

# 45-degree line
fig20.add_trace(go.Scatter(
    x=lims, y=lims, mode='lines',
    line=dict(color='#999', width=1.5, dash='dash'),
    showlegend=False, hoverinfo='skip',
))

# ±1 RMSE bands
rmse = np.sqrt(np.mean(resid**2))
for sign, lbl in [(1, '+1 RMSE'), (-1, '\u22121 RMSE')]:
    fig20.add_trace(go.Scatter(
        x=lims, y=[y + sign * rmse for y in lims], mode='lines',
        line=dict(color='rgba(74,111,165,0.3)', width=1.2, dash='dot'),
        name=lbl, showlegend=True, hoverinfo='skip',
    ))

# Points colored by cluster
top5_idx = set(np.argsort(np.abs(resid))[::-1][:5])
codes  = sub3b['Country Code'].values
names  = sub3b['Country Name'].values
cls    = sub3b['ClusterLabels'].values
years  = sub3b['Year'].values

for cl_lbl in sorted(set(cls)):
    color = CLUSTER_COLORS.get(cl_lbl, '#999')
    mask  = (cls == cl_lbl) & np.array([i not in top5_idx for i in range(len(y_actual))])
    if mask.sum() == 0: continue
    fig20.add_trace(go.Scatter(
        x=y_actual[mask], y=y_pred[mask], mode='markers',
        marker=dict(size=5, color=color, opacity=0.65, line=dict(color='white', width=0.5)),
        name=cl_lbl,
        customdata=np.stack([codes[mask], names[mask], years[mask]], axis=1),
        hovertemplate='<b>%{customdata[1]}</b> (%{customdata[0]}, %{customdata[2]})<br>'
                      'Actual: %{x:.3f}<br>Pred: %{y:.3f}<extra></extra>',
    ))

# Outlier labels
out_idx = list(top5_idx)
fig20.add_trace(go.Scatter(
    x=y_actual[out_idx], y=y_pred[out_idx], mode='markers+text',
    marker=dict(size=9, color='#d4853b', opacity=0.9, line=dict(color='white', width=1.2)),
    text=codes[out_idx], textposition='top center', textfont=dict(size=8, color='#555'),
    name='Largest residuals',
    hovertemplate='<b>%{text}</b><br>Actual: %{x:.3f}<br>Pred: %{y:.3f}<extra></extra>',
))

r2_val = res3b.rsquared
fig20.add_annotation(
    x=0.05, y=0.94, xref='paper', yref='paper',
    text=f'<b>R\u00b2 = {r2_val:.3f}  RMSE = {rmse:.3f}</b>',
    showarrow=False, font=dict(size=12, color=NAVY),
    bgcolor='rgba(255,255,255,0.85)', borderpad=4,
)

fig20.update_layout(**base_layout(
    height=580,
    margin=dict(l=70, r=40, t=55, b=70),
    xaxis=dict(title='Actual ECI', range=lims, gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Predicted ECI (Model 3b)', range=lims, gridcolor=GRID, gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=10)),
))
save(fig20, '20_reg_predicted_vs_actual', w=1100, h=580)
fig20.show(config=CFG)


## Done

All outputs saved to `Visualisation/charts/regression/`.

In [ ]:
print('=' * 60)
print('Regression Visualisations complete.')
print('Charts saved to:', OUT)
for c in ['16_reg_coefficient_forest', '17_reg_r2_comparison',
          '18_reg_hci_production_interaction', '19_reg_nr_rents_vs_eci',
          '20_reg_predicted_vs_actual']:
    exists = os.path.exists(os.path.join(OUT, c + '.html'))
    print(f'  {c}.html  [{"OK" if exists else "MISSING"}]')
